1. Introduction and Business Value

###Project Background

Small businesses increasingly use AI-generated advertising content because it reduces the time and cost of campaign creation. However, the quality and business impact of AI-generated content can vary substantially across platforms, audiences, and content types. Without a structured analytics workflow, it is difficult to determine which content performs well, which performance shifts are abnormal, and whether a new content strategy should be adopted.

Business Decision Supported

This project supports the decision of how to evaluate and improve AI-generated advertising content in a platform-like environment. The goal is not only to monitor headline KPIs, but also to identify unusual performance changes, diagnose likely causes, compare experimental variants, and provide evidence for future content optimization decisions.

Project Objective

The objective of this project is to build an end-to-end analytics workflow for AI-generated advertising content, covering:

KPI monitoring,
anomaly detection,
segmented root-cause analysis,
A/B test evaluation,
and the transition toward predictive and prescriptive analytics.

Why It Matters

This project has practical business value because it translates platform data into decision support. Instead of relying on intuition alone, business teams can use measurable indicators to decide which content versions to scale, which segments require attention, and which changes may improve performance while reducing risk.

2. Data Description

###Dataset Overview

Due to privacy and platform confidentiality considerations, this project uses a simulated dataset designed to reflect realistic content-platform behavior. The dataset preserves the structure of advertising KPI monitoring, segmentation analysis, anomaly scenarios, and A/B testing while avoiding the use of sensitive real-world records.

Data Design Rationale

The simulated data was constructed to approximate a small-business advertising environment in which AI-generated content is distributed across different segments and channels. The dataset includes campaign-level or content-level observations and contains both business metrics and experimental indicators.

Representative Variables

The dataset includes variables such as:

content category,
traffic source,
user segment,
platform or channel identifier,
experiment group,
impressions,
clicks,
completion rate,
engagement rate,
report rate,
and conversion rate.

Decision Relevance of the Data

These variables were chosen because they represent different stages of the advertising funnel, from exposure and interaction to risk signals and business outcomes. This makes the dataset suitable for descriptive analysis, anomaly monitoring, and future predictive modeling.

In [ ]:
# Note: This code assumes the dataset is available in the project directory. For full execution, see the project repository.
import pandas as pd

# Load actual dataset
file_path = 'data/raw/content_platform_mock_data.csv'
df = pd.read_csv(file_path)

print('Shape:', df.shape)
print('\nHead:')
print(df.head())
print('\nInfo:')
print(df.info())
print('\nDescribe all:')
print(df.describe(include='all'))

summary = {
    'num_observations': [df.shape[0]],
    'num_variables': [df.shape[1]],
    'num_numeric_variables': [df.select_dtypes(include='number').shape[1]],
    'num_categorical_variables': [df.select_dtypes(include='object').shape[1]],
    'num_time_variables': [len([c for c in df.columns if 'date' in c or 'time' in c])],
    'total_missing_values': [df.isna().sum().sum()],
    'duplicate_rows': [df.duplicated().sum()],
}
summary_df = pd.DataFrame(summary)
print('\nDataset summary table:')
print(summary_df)

The simulated dataset used in this project contains 97,200 observations and 20 variables. It includes 12 numeric variables, 8 categorical variables, and 1 time-related variable. Initial data quality checks show 0 missing values and 0 duplicate rows, which makes the dataset suitable for descriptive analytics, anomaly monitoring, and A/B evaluation.

3. Data Cleaning and Preprocessing

###Data Cleaning Goals

Before conducting KPI monitoring and performance evaluation, the dataset must be checked for completeness, consistency, and plausibility. In this project, data cleaning focuses on preparing a structured analysis table that can support descriptive analytics, anomaly detection, and future model development.

Main Preprocessing Steps

The preprocessing workflow includes:
checking for missing values,
identifying duplicate records,
converting variables to appropriate data types,
validating KPI ranges,
and preparing categorical variables for grouped analysis and later modeling.

KPI Consistency Checks

Because the dataset is designed to represent advertising performance, KPI validity checks are especially important. For example:

rates such as CTR, completion rate, engagement rate, report rate, and conversion rate should fall within valid ranges,
clicks should not exceed impressions,
and conversions should not exceed clicks when the metric logic requires funnel consistency.

Why This Step Matters

These checks improve the credibility of downstream results. They also reduce the risk that later findings are driven by malformed data rather than meaningful business variation.

In [ ]:
# Note: This code assumes the dataset is loaded as df from the previous cell. For full execution, see the project repository.
# Missing value summary
print(df.isna().sum().sort_values(ascending=False))

# Duplicate row check
print('Duplicate rows:', df.duplicated().sum())

# Data type display
print(df.dtypes)

# KPI validity and funnel checks
kpi_checks = pd.DataFrame({
    'check': ['click<=exposure', 'view_complete<=click', 'conversion_cnt<=click'],
    'violations': [
        (df['click'] > df['exposure']).sum(),
        (df['view_complete'] > df['click']).sum(),
        (df['conversion_cnt'] > df['click']).sum(),
    ]
})
print(kpi_checks)

invalid_rows = df[(df['click'] > df['exposure']) | (df['view_complete'] > df['click']) | (df['conversion_cnt'] > df['click'])]
print('Invalid row sample:', invalid_rows.head())

4. Outlier Detection and Data Quality Checks

In [ ]:
###Why Outlier Detection Is Important

In advertising and platform analytics, unusual KPI values may indicate either data quality problems or meaningful business events. A sudden increase in CTR, an unexpected drop in conversion rate, or a spike in report rate may reflect abnormal traffic, segment mismatch, experiment effects, or simulated business shocks. Therefore, outlier detection is an essential part of both monitoring and analysis.

Methods Used

This project applies simple and interpretable methods for outlier detection, including:

distribution-based checks such as the IQR rule for extreme values,
and time-based monitoring to identify unusual KPI movements over time.

These methods are appropriate for a monitoring-oriented project because they are transparent, easy to explain, and useful for both diagnosis and communication.

Interpretation Strategy

Outliers are not automatically removed. Instead, they are first flagged and interpreted in context. In a platform setting, an outlier may be a real signal rather than an error. For this reason, flagged observations are used to support further diagnosis by content category, traffic source, or user segment.

Summary of Findings

Preliminary monitoring suggests that a small number of observations or time periods show abnormal KPI behavior. These cases provide useful examples for demonstrating how anomaly flags can trigger deeper segmented analysis rather than simple reporting.

In [ ]:
# Note: This code assumes the dataset is loaded as df. For full execution, see the project repository.
df['date'] = pd.to_datetime(df['date'])
daily = df.groupby('date', as_index=False).agg(
    exposure=('exposure', 'sum'),
    click=('click', 'sum'),
    view_complete=('view_complete', 'sum'),
    conversion_cnt=('conversion_cnt', 'sum'),
)
daily['ctr'] = daily['click'] / daily['exposure'].clip(lower=1)

# IQR-based outlier on CTR
q1 = daily['ctr'].quantile(0.25)
q3 = daily['ctr'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
daily['ctr_outlier'] = (daily['ctr'] < lower) | (daily['ctr'] > upper)

outliers = daily[daily['ctr_outlier']]
print('CTR outliers:')
print(outliers)

# Chart
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.plot(daily['date'], daily['ctr'], marker='o', label='CTR')
plt.scatter(outliers['date'], outliers['ctr'], color='red', label='Outlier', zorder=3)
plt.axhline(lower, color='grey', linestyle='--', label='Lower bound')
plt.axhline(upper, color='grey', linestyle='--', label='Upper bound')
plt.title('Daily CTR with IQR Outlier Flags')
plt.xlabel('Date')
plt.ylabel('CTR')
plt.legend()
plt.show()

outlier_table = outliers[['date', 'ctr', 'exposure', 'click', 'ctr_outlier']]
print(outlier_table)

5. Descriptive Analytics and A/B Evaluation

###Overview

The descriptive analytics component summarizes how AI-generated advertising content performs across key business indicators. Rather than relying on a single success metric, this project uses a KPI framework covering exposure, interaction, risk, and conversion signals.

KPI Framework

The core KPIs include:

impressions,
click-through rate,
completion rate,
engagement rate,
report rate,
and conversion rate.

Together, these metrics capture different parts of the content-performance funnel and allow a more balanced assessment of effectiveness.

Trend Monitoring

Trend monitoring is used to observe whether KPI behavior is stable or changing over time. This is important because performance shifts may reflect either normal campaign variation or emerging problems that require attention.

Segmented Analysis

To move beyond aggregate reporting, performance is also examined across business segments such as content category, traffic source, and user group. This helps identify whether a KPI change is broad-based or concentrated in a particular subgroup.

A/B Evaluation

The project also includes an A/B test evaluation module to compare alternative content variants or strategy settings. The purpose is to determine whether a treatment condition improves performance relative to a baseline while also checking for possible trade-offs across multiple KPIs.

Analytical Value

This stage of the project demonstrates that business monitoring is most useful when KPI tracking, anomaly diagnosis, and experiment comparison are connected into one workflow rather than treated as separate tasks.

The A/B test summary suggests that group B outperformed group A on both CTR and conversion rate, indicating that the treatment condition may have improved content performance.

At the aggregate level, the dataset shows 160,653,506 total exposures, 16,508,371 clicks, and 1,185,990 conversions, corresponding to an overall CTR of about 10.28% and a conversion rate of about 7.18%.

The dashboard prototype provides four main analytical components: KPI monitoring, trend monitoring, anomaly alerts, segmented diagnosis, and A/B test comparison. In this notebook, these dashboard outputs are summarized in a notebook-friendly format for PDF submission.

In [ ]:
# Note: This code assumes the dataset is loaded as df. For full execution, see the project repository.
import plotly.express as px

# KPI overview
kpi_summary = df.agg(
    exposure=('exposure','sum'),
    click=('click','sum'),
    view_complete=('view_complete','sum'),
    conversion_cnt=('conversion_cnt','sum'),
)
kpi_summary['ctr'] = kpi_summary['click'] / max(kpi_summary['exposure'],1)
kpi_summary['conversion_rate'] = kpi_summary['conversion_cnt'] / max(kpi_summary['click'],1)
print(kpi_summary)

# Trend chart
daily2 = df.groupby('date', as_index=False).agg(exposure=('exposure','sum'), click=('click','sum'))
daily2['ctr'] = daily2['click'] / daily2['exposure'].clip(lower=1)
fig = px.line(daily2, x='date', y='ctr', title='Daily CTR Trend')
fig.show()

# Segmented performance by content_category
seg = df.groupby('content_category', as_index=False).agg(
    exposure=('exposure','sum'),
    click=('click','sum'),
    conversion_cnt=('conversion_cnt','sum')
)
seg['ctr'] = seg['click'] / seg['exposure'].clip(lower=1)
fig2 = px.bar(seg, x='content_category', y='ctr', title='CTR by Content Category')
fig2.show()

# A/B summary
a_b_summary = df.groupby('experiment_group').agg(exposure=('exposure','sum'), click=('click','sum'), conversion_cnt=('conversion_cnt','sum'))
a_b_summary['ctr'] = a_b_summary['click'] / a_b_summary['exposure'].clip(lower=1)
a_b_summary['conversion_rate'] = a_b_summary['conversion_cnt'] / a_b_summary['click'].clip(lower=1)
print(a_b_summary)

fig3 = px.bar(a_b_summary.reset_index(), x='experiment_group', y=['ctr','conversion_rate'], barmode='group', title='A/B experiment group CTR and Conversion Rate')
fig3.show()

# significance test
import numpy as np
from scipy.stats import norm
ab=df.groupby(['experiment_group','date']).agg(exposure=('exposure','sum'), click=('click','sum')).reset_index()
A=ab[ab['experiment_group']=='A']; B=ab[ab['experiment_group']=='B']
A_ctr=A['click'].sum()/A['exposure'].sum(); B_ctr=B['click'].sum()/B['exposure'].sum()
se=np.sqrt(A_ctr*(1-A_ctr)/A['exposure'].sum()+B_ctr*(1-B_ctr)/B['exposure'].sum())
z=(B_ctr-A_ctr)/se; p=2*(1-norm.cdf(abs(z)))
print(f'A CTR={A_ctr:.4f}, B CTR={B_ctr:.4f}, z={z:.3f}, p={p:.4f}')

6. Model Update

###Current Modeling Status

At the current stage, the project has established the descriptive analytics foundation, including KPI monitoring, anomaly diagnosis, segmented performance analysis, and A/B evaluation. The next analytical layer is predictive modeling, which will use structured features to estimate content performance outcomes.

Planned Predictive Objective

The predictive component is intended to model content performance using variables such as content type, traffic source, user segment, and experimental context. Depending on the final design, the target variable may be defined as:

a continuous KPI such as engagement rate or conversion rate,
or a binary label representing high-performing versus low-performing content.

Planned Models

The initial modeling plan includes:

interpretable baseline models such as linear regression or logistic regression,
followed by more flexible non-linear models such as random forest or gradient-boosted trees.

This progression is intended to balance interpretability and predictive performance.

Evaluation Metrics

Model performance will be evaluated using task-appropriate metrics. For regression tasks, likely metrics include RMSE, MAE, and R-squared. For classification tasks, likely metrics include precision, recall, F1-score, and ROC-AUC. Cross-validation will be used to improve the robustness of the evaluation.

Connection to Decision Support

The purpose of the predictive stage is not only to estimate outcomes, but also to support later prescriptive recommendations. In other words, the model will help identify which content characteristics are associated with better expected performance and which combinations may require revision.

7. Next Steps and Timeline

###Planned Next Steps

The next phase of the project focuses on extending the current descriptive analytics workflow into predictive and prescriptive decision support.

Short-Term Priorities

finalize the cleaned analysis dataset,
confirm the primary modeling target,
engineer structured features from content and segment information,
train baseline predictive models,
compare model performance across candidate methods,
and translate model outputs into actionable content strategy recommendations.

Proposed Timeline

Week 1: finalize dataset structure, complete preprocessing checks, and confirm KPI definitions.
Week 2: build baseline predictive models and evaluate initial performance.
Week 3: refine model selection, interpret results, and identify the most relevant content-performance drivers.
Week 4: connect descriptive findings and predictive outputs into a prescriptive recommendation workflow and finalize the project presentation materials.

Expected Improvements

The project can be improved further by:

selecting one primary business target more explicitly,
validating model robustness across segments,
and making the link between model output and recommendation logic more operational.

8. Conclusion

This project develops a decision-support workflow for evaluating and improving AI-generated advertising content in a simulated small-business platform environment. So far, the project has established the descriptive analytics foundation through KPI monitoring, anomaly detection, segmented analysis, and A/B evaluation. The next step is to extend this workflow into predictive and prescriptive analytics so that the system can move from performance observation to actionable content strategy guidance.

A separate appendix report generated from the running dashboard is provided as supplementary material to document the full KPI summary, A/B test outputs, and supporting tables.Due to environment limitations in the browser-based notebook interface, the main analytical workflow and project interpretation are presented in this notebook, while the full dashboard-generated outputs are included in a separate appendix file.